# 因子构造 
以均值MA和标准差SA作为双因子，构建双因子模型。  

 

## 导入库

In [29]:
import json  
import os
import re
import polars as pl
from pathlib import Path
import warnings

## 超参数

In [30]:
TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

## 读取数据
数据的保存路径为：    

```
DATA_BASE_DIR  
|- TASK_ID1
|   |- Node1
|   |   |- performance_and_record_0.json   
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- Node2
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- ...
|- TASK_ID2
|   |- Node1
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
```  

其中，task_id的构成为 时间_中缀_uuid，例如：20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57  
指定中缀，选取所有中缀一样的task_id，得到所有node的路径  

In [31]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

node_paths


['/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031879_25181',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031891_9646',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031901_11800',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014501_6831',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014595_27084',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014583_23544',
 '/home/frank/files/programs/Graduati

In [32]:
# 获取其下所有performance_and_record_*.jsonl文件的路径
jsonl_paths = []
for node_path in node_paths:
    all_files = os.listdir(node_path)
    perf_and_rewa_jsonl_files = [file for file in all_files if file.endswith('.jsonl') and 'performance_and_reward_' in file]
    jsonl_paths.extend(
        os.path.join(node_path, file)
        for file in perf_and_rewa_jsonl_files
    )
jsonl_paths

['/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_3.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_1.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_4.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031879_25181/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031879_25181/performance_and_reward_

先用scan获取所有的jsonl  
使用polars的concat方法，合并所有json，字段由performance_and_record约定好  
展示测试数据  

In [33]:
lazy_frames = [pl.scan_ndjson(f) for f in jsonl_paths]
lf = pl.concat(lazy_frames)   # 得到 LazyFrame


## 处理数据  
加载的lf，其结构如上所示，需要进行解析   

>- 1.基线回归为单证券，因此，需要验证portfolio中列表长度，如果有长度大于1的列表，需要警告，并且取[0]  
>- 2. data比较复杂，由多项fields组成，包括decision_weights, performance, normalized_performance, reward； 其中，如果是多证券，decision_weights, 是list结构，需要判断是否有长度大于1的列表，如果有，需要警告，并且取[0]; performance 和 normalized_performance是固定len=5的list，对应字段return, -vol, sharp, -maxdrawdwon, devisification，需要展开为对应字段   

In [34]:
# 展示fields
df_demo = lf.head().collect()
df_demo.with_columns(pl.col('data').struct.unnest()).select(pl.all().exclude('data'))

year,month,portfolio,decision_weights,performance,normalized_performance,reward
i64,i64,list[str],list[f64],list[f64],list[f64],f64
2022,1,"[""688071""]","[1.0, 0.0]","[-0.055, -0.0998, … -0.0]","[0.0668, -9.9794, … -0.8893]",-0.0717
2022,1,"[""301087""]","[1.0, 0.0]","[-0.0623, -0.0035, … -0.0]","[0.0151, -0.3536, … -0.8893]",-0.0623
2022,1,"[""603171""]","[1.0, 0.0]","[0.1262, -0.0631, … -0.0]","[1.3609, -6.3076, … -0.8893]",0.1195
2022,1,"[""688257""]","[1.0, 0.0]","[-0.1147, -0.0089, … -0.0]","[-0.3593, -0.885, … -0.8893]",-0.1148
2022,1,"[""688192""]","[1.0, 0.0]","[-0.0116, -0.0593, … -0.0]","[0.3768, -5.9307, … -0.8893]",-0.0175


In [35]:
# 定义字段
perf_fields = ["return", "neg_vol", "sharp", "neg_maxdrawdown", "diversification"]

# 1. 在 LazyFrame 上完成 unnest，再 collect（单次扫描，避免先 collect 再在 Python 里 unnest）
lf_unnested = lf.with_columns(pl.col("data").struct.unnest()).drop("data")
df_flat = lf_unnested.collect()  # 需要子集时可改为 lf_unnested.head(n).collect()

# 2. portfolio：检查是否有多证券，有则告警并只保留 [0]
if (df_flat["portfolio"].list.len() > 1).any():
    warnings.warn("发现 portfolio 列表长度 > 1（多证券），已取 [0] 作为单证券基线。")
df_flat = df_flat.with_columns(pl.col("portfolio").list.get(0).alias("portfolio"))

# 3. decision_weights：多证券时告警并取 [0]
if (df_flat["decision_weights"].list.len() > 2).any():
    warnings.warn("发现 decision_weights 列表长度 > 2，已取 [0]。")
df_flat = df_flat.with_columns(pl.col("decision_weights").list.get(0).alias("decision_weights"))

# 4. performance / normalized_performance：固定 len=5，展开为多列（return, -vol, sharp, -maxdrawdown, diversification）
df_flat = df_flat.with_columns(
    pl.col("performance").list.to_struct(fields=[f"{f}" for f in perf_fields]).struct.unnest()
).drop("performance")
df_flat = df_flat.with_columns(
    pl.col("normalized_performance").list.to_struct(fields=[f"normalized_{f}" for f in perf_fields]).struct.unnest()
).drop("normalized_performance")
# reward 已是标量，无需处理

In [36]:
df_flat.head()

year,month,portfolio,decision_weights,reward,return,neg_vol,sharp,neg_maxdrawdown,diversification,normalized_return,normalized_neg_vol,normalized_sharp,normalized_neg_maxdrawdown,normalized_diversification
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2022,1,"""688071""",1.0,-0.0717,-0.055,-0.0998,-1.5513,-0.2956,-0.0,0.0668,-9.9794,-155.1326,-29.5619,-0.8893
2022,1,"""301087""",1.0,-0.0623,-0.0623,-0.0035,-16.6012,-0.114,-0.0,0.0151,-0.3536,-1660.1161,-11.4,-0.8893
2022,1,"""603171""",1.0,0.1195,0.1262,-0.0631,1.0,-0.0,-0.0,1.3609,-6.3076,99.9984,-0.0,-0.8893
2022,1,"""688257""",1.0,-0.1148,-0.1147,-0.0089,-11.9551,-0.2005,-0.0,-0.3593,-0.885,-1195.5112,-20.0521,-0.8893
2022,1,"""688192""",1.0,-0.0175,-0.0116,-0.0593,-1.1959,-0.1403,-0.0,0.3768,-5.9307,-119.5922,-14.0342,-0.8893


In [37]:
# 转为lf，方便后续处理
lf_flat = df_flat.lazy()

去掉return为0的行  

In [38]:
lf_flat = lf_flat.filter(pl.col('return') != 0)

## 构建因子
为了减少处理难度，先筛选需要的列  

- year   
- month    
- portfolio    
- decision_weights     
- return (表示year-month下一个月的收益)   

In [39]:
needed_cols = ['year','month','portfolio','decision_weights','return']
filtered_lf = lf_flat.select(
    pl.col(needed_cols)
)
filtered_lf.head().collect()

year,month,portfolio,decision_weights,return
i64,i64,str,f64,f64
2022,1,"""688071""",1.0,-0.055
2022,1,"""301087""",1.0,-0.0623
2022,1,"""603171""",1.0,0.1262
2022,1,"""688257""",1.0,-0.1147
2022,1,"""688192""",1.0,-0.0116


构建MA和SA因子 
MA为决策权重的均值，SA为决策权重的标准差倒数  

In [40]:
masa_lf = filtered_lf.group_by(['year','month','portfolio']).agg(
        (pl.col('decision_weights').mean().alias('MA'),
        (1 / pl.col('decision_weights').std()).alias('SA'),
        pl.col('return').first().alias('return')
    )
)
masa_lf.head().collect()

year,month,portfolio,MA,SA,return
i64,i64,str,f64,f64,f64
2024,3,"""688003""",0.857143,4.99886,-0.0371
2024,8,"""603385""",0.719332,3.970937,0.1586
2022,5,"""300811""",0.704216,3.069513,0.0803
2011,6,"""600035""",0.549417,2.700038,-0.0042
2024,6,"""688612""",0.53858,9.332535,-0.0514


最后，将year,month合并为date  

In [41]:
masa_lf = masa_lf.with_columns(
    pl.date(pl.col('year'), pl.col('month'), 1).alias('date')
).drop(['year','month']).select(['date','portfolio','MA','SA','return'])
masa_lf.head().collect()

date,portfolio,MA,SA,return
date,str,f64,f64,f64
2005-09-01,"""600236""",0.689971,2.956386,-0.0578
2024-01-01,"""300478""",0.644491,3.433138,-0.0357
2017-09-01,"""600176""",0.533153,3.051015,0.0035
2020-02-01,"""603040""",0.627056,3.370836,-0.0018
2015-01-01,"""002508""",0.463826,2.966182,0.0645


## 保存数据  
保存数据为parquet文件  

In [42]:
if SAVE:
    dir_path = Path(SAVE_BASELINE_REG_DIR)
    dir_path.mkdir(parents=True, exist_ok=True)
    masa_lf.collect().write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, f'基准回归-MA,SA双因子.parquet'))
    print(f'保存成功，路径为:{os.path.join(SAVE_BASELINE_REG_DIR, f"基准回归-MA,SA双因子.parquet")}')
else:
    print('请设置SAVE=True，以保存数据')

保存成功，路径为:/home/frank/files/programs/GraduationThesis/empirical/baseline1/baseline_reg/基准回归-MA,SA双因子.parquet
